# Seleksi query asli dari Trends dan PAA

Prosedur baru tidak melakukan sintesis atau penerjemahan. Notebook memuat teks asli yang sudah terkumpul, menyaring bahasa, dan menyimpan keputusan kualitas secara terpisah.

**Kandidat bukan query final.** Penandaan bahasa oleh asisten tidak sama dengan penerimaan penelitian. Status pilot Gemini sebelumnya tidak dipakai sebagai alasan menerima query. Jalankan sel dari atas memakai kernel `venv`; tidak ada pemanggilan API.

In [1]:
from pathlib import Path
import sys
import csv
from html import escape
from IPython.display import HTML, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/prepare_original_queries.py').exists())
sys.path.insert(0, str(ROOT / 'src'))
from prepare_original_queries import prepare
records = prepare()

Kemunculan sumber: 78
Kandidat Indonesia: 49 {'kesehatan': 17, 'keuangan': 20, 'teknologi': 12}
Ditunda karena bahasa: 29
Query unik diterima: 41


## Tinjau kandidat

Periksa cakupan domain, kejelasan kebutuhan informasi, kebutuhan konteks tambahan, asumsi dalam pertanyaan, dan kemiripan intent. Query yang meminta akses layanan dapat dipisahkan dari pencarian artikel sesuai aturan yang dibekukan. Ejaan asli dipertahankan.

`deferred_language` bukan penghapusan sumber: teks non-Indonesia masih tersimpan, tetapi belum digunakan dalam jalur query asli berbahasa Indonesia. Gunakan `STATUS=None` untuk melihat semuanya.

In [3]:
DOMAIN = None
STATUS = 'needs_review'  # pending, accepted, needs_review, excluded, deferred_language, atau None
KEYWORD = ''
START = 0
PAGE_SIZE = 20

selected = [r for r in records if (DOMAIN is None or r['domain'] == DOMAIN)
            and (STATUS is None or r['selection_status'] == STATUS)
            and KEYWORD.casefold() in r['query_text'].casefold()]
columns = ['query_text', 'domain', 'source_type', 'language', 'selection_status', 'selection_reason', 'used_in_technical_pilot']
header = ''.join('<th>' + escape(k) + '</th>' for k in columns)
body = ''.join('<tr>' + ''.join('<td>' + escape(str(r[k])) + '</td>' for k in columns) + '</tr>'
               for r in selected[START:START + PAGE_SIZE])
display(HTML('<div style="overflow-x:auto"><table><tr>' + header + '</tr>' + body + '</table></div>'))
print('Kandidat sesuai filter:', len(selected))

query_text,domain,source_type,language,selection_status,selection_reason,used_in_technical_pilot
Sunscreen apa yang cepat menghilangkan flek hitam?,kesehatan,people_also_ask,id,needs_review,Periksa asumsi efek cepat dalam pertanyaan sebelum menerima; teks sumber tidak diubah.,False
Bagaimana cara cek pajak NPWP?,keuangan,people_also_ask,id,needs_review,"Objek pemeriksaan belum jelas: status NPWP, kewajiban, atau pembayaran pajak.",False
Pinjam 5 juta di Pegadaian bunganya berapa?,keuangan,people_also_ask,id,needs_review,Produk dan tenor belum disebutkan; perlu keputusan apakah query ini cukup mandiri untuk penelitian.,False
Berapa dividen 1 lot saham BCA?,keuangan,people_also_ask,id,needs_review,Periode pembagian dividen belum disebutkan; perlu keputusan penanganan konteks waktu.,False
Modal 100 juta dapat dividen berapa?,keuangan,people_also_ask,id,needs_review,Instrumen investasi dan periode tidak tersedia; tidak menambahkan asumsi pada teks asli.,False
Dividen paling besar saham apa?,keuangan,people_also_ask,id,needs_review,"Ukuran terbesar belum jelas, misalnya nominal atau yield; perlu pemeriksaan intent.",False
hakikat senam aerobik menurut jackie sorensens,kesehatan,google_trends,id,needs_review,Periksa kesesuaian cakupan kebutuhan informasi kesehatan dibanding pertanyaan materi akademik.,False


Kandidat sesuai filter: 7


## Keputusan berdasarkan teks

Cukup salin **teks query** dari tabel ke dictionary; tidak perlu mencari ID. `accepted` berarti layak sebagai query sumber menurut penilaian kualitas, belum berarti layak grounding atau siap pemodelan. `needs_review` untuk kasus yang belum jelas; `excluded` harus diberi alasan.

Contoh komentar di bawah tidak dijalankan. Anda dapat meminta asisten mengisi keputusan berdasarkan arahan atau kriteria yang disepakati. Dictionary kosong mempertahankan progres tersimpan.

In [ ]:
decisions_by_text = {
    # 'Apa penyebab penyakit campak?': {
    #     'status': 'accepted',
    #     'reason': 'Pertanyaan Indonesia, sesuai kesehatan, dan kebutuhan informasi jelas.',
    # },
}

changes = {}
for text, decision in decisions_by_text.items():
    matches = {r['query_id'] for r in records if r['query_text'] == text}
    if len(matches) != 1:
        raise ValueError(f'Teks tidak ditemukan atau ambigu antar-domain: {text}')
    changes[next(iter(matches))] = decision

records = prepare(changes)
print('Keputusan dan daftar kerja sudah disimpan.')

## Berkas hasil dan pekerjaan berikutnya

- `data/interim/original_queries/source_pool.csv`: seluruh sumber, termasuk yang ditunda karena bahasa.
- `candidates_id.csv`: kandidat yang memenuhi aturan bahasa Indonesia; status kualitas terlihat pada setiap baris.
- `deferred_language.csv`: sumber Inggris/unknown yang tidak diterjemahkan.
- `accepted_unique.csv`: daftar query unik yang telah diterima berdasarkan keputusan kualitas.
- `data/manual/original_query_decisions.csv`: keputusan yang dibaca kembali pada sesi berikutnya.

Masukan saat ini mencakup PAA dan sumber Trends yang sebelumnya ditandai cukup spesifik (`ready_for_synthesis`, nama status lama). Topik Trends `needs_paa` tetap antre untuk pencarian tambahan; belum otomatis menjadi kandidat query siap pakai.

Setelah keputusan kualitas selesai, pilih batch pengumpulan Google Top-10 dan Gemini dengan konfigurasi yang dibekukan. Jangan memakai informasi berhasil/tidaknya grounding pilot sebagai satu-satunya dasar menerima query. Perluas sumber menggunakan topik Trends lain bila jumlah atau keragaman kurang.